# Optuna hyperparameter optimization

### Import libraries and set configs

In [1]:
import json
import pandas as pd

import optuna

from cfg.config import SHARED


class CFG:
    n_trials = 250
    # maximum number of simultaneously opened trades for backtest metric
    max_num_simult_trades = SHARED.max_num_simult_trades
    # significance level, that is used to conduct a t-test between 2 models
    optimize_alpha = 0.2
    n_repeats = 1
    n_folds = 8
    min_precision = SHARED.min_precision
    TP = SHARED.TP
    SL = SHARED.SL
    slippage = SHARED.slippage
    test_time_days = 90

/home/alex/Repos/sigbot/.venv/lib/python3.12/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load the train data

In [2]:
train_df = pd.read_pickle("data/train_df.pkl")

# all data for the last 90 days are test
test_date = train_df["time"].max() - pd.to_timedelta(CFG.test_time_days, unit="D")

fi = pd.read_csv("model/features/feature_importance.csv")

### Optimize

In [ ]:
from utils.optimization_utils import make_objective

with open("model/bybit_tickers.json", "r") as f:
    bybit_tickers = json.load(f)

objective = make_objective(
    train_df, 
    test_date, 
    fi, 
    bybit_tickers, 
    TP=CFG.TP - CFG.slippage,
    SL=CFG.SL + CFG.slippage,
    n_folds=CFG.n_folds, 
    optimize_alpha=CFG.optimize_alpha, 
    min_precision=CFG.min_precision,
)

study = optuna.create_study(
    direction="maximize",
    study_name="lgbm",
    storage="sqlite:///model/optuna/optuna_lgbm.db",
    load_if_exists=True,   # resume instead of erroring if the study already exists
)
remaining = CFG.n_trials - len(study.trials)
study.optimize(objective, n_trials=max(remaining, 0))

print("Number of finished trials: {}".format(len(study.trials)))

print("Best trial:")
trial = study.best_trial

print("  Value: {}".format(trial.value))

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

df_optuna = study.trials_dataframe()
df_optuna = df_optuna.sort_values("value", ascending=False)
df_optuna.to_csv("optuna/optuna_lgbm.csv", index=False)

display(df_optuna.head(10))

/home/alex/Repos/sigbot/.venv/lib/python3.12/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[I 2026-07-18 13:35:02,857] Using an existing study with name 'lgbm' instead of creating a new one.
